In [28]:
from dataclasses import dataclass
from decimal import Decimal


@dataclass(frozen=True)
class Dinheiro:
    valor: Decimal

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError("Valor não pode ser negativo")

    def somar(self, outro: "Dinheiro") -> "Dinheiro":
        return Dinheiro(self.valor + outro.valor)


@dataclass(frozen=True)
class Item:
    nome: str
    preco: Dinheiro


class Itens:
    def __init__(self, itens: list[Item]):
        self._itens = itens

    def total(self) -> Dinheiro:
        return sum(
            (i.preco for i in self._itens),
            Dinheiro(Decimal(0))
        )

    def __iter__(self):
        return iter(self._itens)


class Pedido:
    def __init__(self, itens: Itens):
        self._itens = itens
        self._status = "aberto"

    def pagar(self):
        if self._status != "aberto":
            raise ValueError("Pedido já foi processado")
        self._status = "pago"

    def total(self) -> Dinheiro:
        return self._itens.total()

    def esta_pago(self) -> bool:
        return self._status == "pago"

In [34]:
cafe = Item("Café", Dinheiro(Decimal("5.50")))
pao  = Item("Pão",  Dinheiro(Decimal("2.00")))

itens = Itens([cafe, pao])

# Criando pedido
pedido = Pedido(itens)



pedido.pagar()

print(pedido.esta_pago()) # True


True


Explicação completa do código

from dataclasses import dataclass dataclass é um decorador do Python (3.7+) que automaticamente gera métodos boilerplate como `__init__ `, `__repr__` e `__eq__` baseado nos atributos declarados na classe.

In [6]:
# Sem dataclass — você escreve tudo na mão
class Ponto:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"Ponto(x={self.x}, y={self.y})"
    def __eq__(self, outro):
        return self.x == outro.x and self.y == outro.y

# Com dataclass — Python gera tudo isso automaticamente
@dataclass
class Ponto:
    x: float
    y: float

from decimal import Decimal

float tem problemas de precisão em dinheiro por ser binário:

In [11]:
0.1 + 0.2
0.30000000000000004  # ❌ erro de ponto flutuante

Decimal("0.1") + Decimal("0.2")
Decimal('0.3')       # ✅ preciso

Decimal('0.3')

In [7]:
0.1 + 0.2

0.30000000000000004

In [9]:
Ponto(0.1 , 0.2)

Ponto(x=0.1, y=0.2)

Por isso nunca use float para valores monetários — use Decimal.

A classe Dinheiro

In [12]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True)
class Dinheiro:
    valor: Decimal

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError("Valor não pode ser negativo")

    def somar(self, outro: "Dinheiro") -> "Dinheiro":
        return Dinheiro(self.valor + outro.valor)

@dataclass(frozen=True) — o frozen=True torna o objeto imutável. Nenhum atributo pode ser reatribuído após a criação. Isso é importante para um Value Object como dinheiro:

In [13]:
d = Dinheiro(Decimal("10"))

In [15]:
d

Dinheiro(valor=Decimal('10'))

In [14]:
d.valor = Decimal("999")  # ❌ lança FrozenInstanceError

FrozenInstanceError: cannot assign to field 'valor'

`__post_init__` — é executado automaticamente após o `__init__` gerado pelo @dataclass. É o lugar certo para validações:

In [16]:
Dinheiro(Decimal("-5"))  # ❌ lança ValueError

ValueError: Valor não pode ser negativo

In [17]:
Dinheiro(Decimal("10"))  # ✅

Dinheiro(valor=Decimal('10'))

somar — retorna um novo objeto Dinheiro em vez de modificar o atual (imutabilidade). A string "Dinheiro" entre aspas é uma forward reference — necessária quando o tipo referencia a própria classe:

In [18]:
d1 = Dinheiro(Decimal("10"))
d2 = Dinheiro(Decimal("5"))
d3 = d1.somar(d2)  # Dinheiro(valor=Decimal('15'))

##### A classe Item

In [19]:
@dataclass(frozen=True)
class Item:
    nome: str
    preco: Dinheiro

Simples e imutável. Um item tem nome e preço. O @dataclass gera `__init__` e `__eq__` automaticamente:

In [20]:
item = Item(nome="Café", preco=Dinheiro(Decimal("5.50")))
print(item)  # Item(nome='Café', preco=Dinheiro(valor=Decimal('5.50')))

Item(nome='Café', preco=Dinheiro(valor=Decimal('5.50')))


In [21]:
Item("Café", Dinheiro(Decimal("5"))) == Item("Café", Dinheiro(Decimal("5")))  # True

True

A classe Itens — Coleção de Primeira Classe

In [22]:
class Itens:
    def __init__(self, itens: list[Item]):
        self._itens = itens

    def total(self) -> Dinheiro:
        return sum(
            (i.preco for i in self._itens),
            Dinheiro(Decimal(0))
        )

    def __iter__(self):
        return iter(self._itens)

Por que não usar list diretamente? Encapsular a coleção permite colocar comportamento nela — como calcular o total.

sum(..., Dinheiro(Decimal(0))) — o segundo argumento do sum é o valor inicial (necessário porque sum começa do 0 inteiro por padrão, e não de um Dinheiro). Funciona porque Python chama valor_inicial + próximo_item internamente... mas atenção: aqui usamos o sum com o gerador e valor inicial do tipo certo:

In [23]:
# O que o sum faz internamente:
# Dinheiro(0) + item1.preco + item2.preco + ...
# Mas Dinheiro não tem __add__, tem .somar()!

⚠️ Bug sutil: o sum usa +, mas Dinheiro só tem .somar(). Para funcionar corretamente, seria necessário implementar __add__:

In [24]:
# Correção
@dataclass(frozen=True)
class Dinheiro:
    valor: Decimal

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError("Valor não pode ser negativo")

    def __add__(self, outro: "Dinheiro") -> "Dinheiro":
        return Dinheiro(self.valor + outro.valor)

    def somar(self, outro: "Dinheiro") -> "Dinheiro":
        return self.__add__(outro)

__iter__ — permite iterar sobre Itens como se fosse uma lista:

In [ ]:
for item in itens:   # funciona por causa do __iter__
    print(item.nome)

A classe Pedido

In [26]:
class Pedido:
    def __init__(self, itens: Itens):
        self._itens = itens
        self._status = "aberto"

    def pagar(self):
        if self._status != "aberto":
            raise ValueError("Pedido já foi processado")
        self._status = "pago"

    def total(self) -> Dinheiro:
        return self._itens.total()

    def esta_pago(self) -> bool:
        return self._status == "pago"

_status com underscore — convenção Python para atributo "privado". Não há privacidade real, mas sinaliza que não deve ser acessado diretamente de fora.

pagar() — aplica a regra 9 do Calisthenics: Tell, don't ask. Em vez de alguém de fora verificar o status e mudá-lo, o próprio Pedido controla sua transição de estado.

esta_pago() — expõe uma pergunta de comportamento, não o estado bruto. A diferença:

In [ ]:
# ❌ expõe estado interno
pedido._status == "pago"

# ✅ expõe comportamento
pedido.esta_pago()

Fluxo completo

In [ ]:
# Criando itens
cafe = Item("Café", Dinheiro(Decimal("5.50")))
pao  = Item("Pão",  Dinheiro(Decimal("2.00")))

itens = Itens([cafe, pao])

# Criando pedido
pedido = Pedido(itens)

print(pedido.total())     # Dinheiro(valor=Decimal('7.50'))
print(pedido.esta_pago()) # False

pedido.pagar()
print(pedido.esta_pago()) # True

pedido.pagar()            # ❌ ValueError: Pedido já foi processado
```

---

### Resumo visual
```
Dinheiro        → Value Object imutável, encapsula Decimal
Item            → Value Object imutável, tem nome e Dinheiro
Itens           → Coleção de primeira classe, sabe calcular total
Pedido          → Entidade com estado e comportamento